<a href="https://colab.research.google.com/github/esla-boom/devf/blob/main/Copia_de_AngelV_Hands_On_Fundamentos_de_LLMs_con_Modelos_Llama.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# **HANDS - ON: FUNDAMENTOS DE LLMS CON MODELOS LLAMA**


Una vez vista la masterclass ***Fundamentos de LLMs y Arquitectura de Llama***, se proporciona el siguiente ***Colab*** para construir, en vivo, la primera llamada a Llama y medir su comportamiento.

Usamos **Groq** para tener acceso a inferencia de Llama con GPU gratuita, sin necesidad de infraestructura propia.

De igual manera, se proporciona la **solución** de este notebook a través del siguiente [enlace](https://colab.research.google.com/drive/1bjMHw7fmMxo9dyQp140e9mqotKRkfKBg?usp=sharing).

## **CONFIGURACIÓN DEL ENTORNO**

Antes de llamar a Llama necesitamos activar la GPU gratuita de Colab y conectar nuestra cuenta de **Groq**. La librería `groq` es el cliente oficial en Python para hacer llamadas a la API.

### **COLAB SECRETS**

Para no exponer tu ***API key*** directamente en el código, Colab ofrece un panel de ***Secrets*** (ícono de llave en la barra lateral izquierda) donde puedes guardarla de forma segura, añadiendo un nombre asociado a la ***API key*** para guardarla dentro de una variable y usarla dentro del notebook.

In [ ]:
# Instalar cliente de Groq y leer API key desde Colab Secrets
!pip install groq -q

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 143.7/143.7 kB 3.8 MB/s eta 0:00:00


In [ ]:
import os
from groq import Groq
from google.colab import userdata

client = Groq(api_key=userdata.get('GROQ_API_KEY'))
print('Cliente de Groq inicializado correctamente.')

Cliente de Groq inicializado correctamente.


## **DE PALABRAS A TOKENS**

Antes de que Llama genere una sola palabra, tu prompt se divide en **tokens**. Vamos a definir un prompt de ejemplo que después enviaremos a Llama.

In [ ]:
# Definir un prompt de ejemplo (una pregunta real de tu propio contexto)
prompt = "¿Cual es la diferencia entre metodologías ágiles y de cascada?"
print(prompt)


¿Cual es la diferencia entre metodologías ágiles y de cascada?


### **PRIMERA LLAMADA A LLAMA (ZERO-SHOT)**

Un LLM predice el siguiente token más probable, **no busca la respuesta en una base de datos**. Vamos a enviar el prompt a Llama en modo zero-shot (sin ejemplos previos) y ver la respuesta generada.

In [ ]:
# Enviar el prompt a Llama en Groq y mostrar la respuesta generada
response = client.chat.completions.create(
    model="allam-2-7b",
    messages=[
        {
            "role": "user",
            "content": prompt
        }
    ]
)
print(response.choices[0].message.content)

La diferencia entre metodologías ágiles y Tradicionales (o Cascada) يكمن أساسا en la forma en que se planifican, desarrollan y llevan a cabo proyectos y en cómo se adaptan a las cambiantes necesidades y requisitos. Estos términos se relacionan con dos paradigmas principales para dirigir proyectos de desarrollo de software.

Metodologías ágiles y Tradicionales difieren de manera clara en las siguientes características:

1. Planning y estimación:
   - Metodologías ágiles: Se hace un esfuerzo consciente para mantener el flujo de trabajo planificado y adaptable, manteniendo el proceso sin planificar fuertemente y retazos por retazo. Por lo tanto, la estimación de tiempos y costos se realize en lo posible de manera heurística y se revisa continuamente.
   - Metodologías Tradicionales: Se centran en una gran planificación anticipada, lo que suele llevar a que tenga que hacerse una gran estimación anticipada del trabajo envolviendo todas las personas y tiempos pertinentes. Esta estimación se 

In [ ]:
print(response.model_dump_json(indent=2))

{
  "id": "chatcmpl-bac2f6a5-e177-4bd0-a6c5-59c31736e1e9",
  "choices": [
    {
      "finish_reason": "stop",
      "index": 0,
      "logprobs": null,
      "message": {
        "content": "La diferencia entre metodologías ágiles y Tradicionales (o Cascada) يكمن أساسا en la forma en que se planifican, desarrollan y llevan a cabo proyectos y en cómo se adaptan a las cambiantes necesidades y requisitos. Estos términos se relacionan con dos paradigmas principales para dirigir proyectos de desarrollo de software.\n\nMetodologías ágiles y Tradicionales difieren de manera clara en las siguientes características:\n\n1. Planning y estimación:\n   - Metodologías ágiles: Se hace un esfuerzo consciente para mantener el flujo de trabajo planificado y adaptable, manteniendo el proceso sin planificar fuertemente y retazos por retazo. Por lo tanto, la estimación de tiempos y costos se realize en lo posible de manera heurística y se revisa continuamente.\n   - Metodologías Tradicionales: Se centran 

### **CUÁNTOS TOKENS CONSUMIÓ EL PROMPT**

La respuesta de la API trae un resumen (`usage`) con el número de tokens de entrada y de salida. Es la forma más directa de responder: ¿cuántos tokens consumió mi prompt?

In [ ]:
# Mostrar cuántos tokens tuvo el prompt y cuántos tuvo la respuesta
print('Tokens del prompt: ', response.usage.prompt_tokens)
print('Tokens de la respuesta: ', response.usage.completion_tokens)
print('Tokens totales: ', response.usage.total_tokens)

Tokens del prompt:  29
Tokens de la respuesta:  1370
Tokens totales:  1399


### **MIDIENDO LA LATENCIA DE INFERENCIA**

Elegir el tamaño correcto de modelo es una decisión de ingeniería, no solo de potencia bruta. Vamos a medir cuánto tarda Llama en responder el mismo prompt.

In [ ]:
# Medir el tiempo de respuesta de Llama para el mismo prompt
import time
start_time = time.time()
response = client.chat.completions.create(
    model="meta-llama/llama-prompt-guard-2-22m",
    messages=[
        {
            "role": "user",
            "content": prompt
        }
    ]
)
end_time = time.time()
duracion = end_time - start_time
print(f"Tiempo de ejecución: {duracion:.2f} segundos")

Tiempo de ejecución: 0.15 segundos


### **COMPARANDO DOS TAMAÑOS DE MODELO**

Repite la misma llamada usando la versión más grande de Llama disponible en Groq y compara tiempo de respuesta y calidad contra el modelo ligero.

In [ ]:
# Repetir la llamada con un modelo Llama más grande y comparar tiempo y calidad
start_time = time.time()
response_grande = client.chat.completions.create(
    model="allam-2-7b",
    messages=[
        {
            "role": "user",
            "content": prompt
        }
    ]
)
end_time = time.time()
duracion_grande = end_time - start_time
print(f"Tiempo de ejecución modelo ligero: {duracion:.2f} s - {response.usage.total_tokens} tokens")
print(f"Tiempo de ejecución modelo grande: {duracion_grande:.2f} s - {response_grande.usage.total_tokens} tokens")
print("\nRespuesta del modelo grande:\n", response_grande.choices[0].message.content)

Tiempo de ejecución modelo ligero: 0.15 s - 0 tokens
Tiempo de ejecución modelo grande: 1.36 s - 1425 tokens

Respuesta del modelo grande:
 Las metodologías en software development (procesos de software) pueden dividirse en dos ampliamente diferenciadas: las metodologías agiles y las metodologías estándar. ¿Has trabajado nunca en ambos tipos de proyectos en el pasado? ¿Qué te parece trabajar en cada uno de los dos tipos de proyectos? 

Una metodología ágila es un tipo de proceso de software utilizado para crear productos o servicios en un continuo avance y remodelamiento, basado en las necesidades de los co-investigadores y los usuarios, sea éstos de negocio o de técnicos. Los aspectos básicos de una metodología ágila son que no existe plano de proyecto escrito, el uso frecuente de equipos pequeños y autonomos, estrategias de interacción regular y constante con el cliente para garantizar que la solución satisfaga sus necesidades, y el uso de prototipos o revisiones intermedias para eva

## **CHALLENGE: COMPARADOR DE MODELOS LLAMA**

Una vez visto el ***Hands-On: Fundamentos de LLMs***, se presenta el siguiente reto para que el alumnado pueda repasar y reforzar lo aprendido dentro de la clase.

Se construirá un pequeño **comparador que envíe 3 preguntas** reales a Llama y registre, para cada una, el **tiempo de respuesta** y el **número de tokens** consumidos.

**IMPORTANTE:** Para su revisión, **es indispensable que los apartados anteriores se encuentren llenados con el código visto durante la sesión.**

### **INSTRUCCIONES:**

**1. Carga la API key y genera 3 preguntas:**

   * Lee la API key de llama ya configurada desde **Colab Secrets**.
   * Construye una lista vacía llamada `preguntas` y agrégale 3 preguntas frecuentes de tu propio contexto (soporte técnico, tienda, escuela, etc.).

In [ ]:
# Leer API key desde Colab Secrets
import os
import time
from groq import Groq
from google.colab import userdata

client = Groq(api_key=userdata.get('GROQ_API_KEY'))
print('Cliente de Groq inicializado correctamente.')

Cliente de Groq inicializado correctamente.


In [ ]:
# Definir la lista de preguntas
preguntas = ["¿Cuáles son los tres roles principales en Scrum y cuál es la responsabilidad de cada uno?",
"¿Cuál es la diferencia entre un Sprint Backlog y un Product Backlog?",
"¿Qué objetivo tiene la Daily Scrum y quiénes deberían participar en ella?"]

In [ ]:
modelos = {
    "modelo_ligero" : "allam-2-7b",
    "modelo_pesado" : "openai/gpt-oss-20b"
}

**2. Consulta la primera pregunta:** Envía la primera pregunta a Llama usando el modelo más ligero disponible en Groq. Guarda la respuesta, el tiempo de respuesta y el número de tokens en un diccionario llamado `resultado_1`.

In [ ]:
# Consultar la primera pregunta y guardar el resultado en resultado_1
start_time = time.time()
response = client.chat.completions.create(
    model=modelos.get("modelo_ligero"),
    messages=[
        {
            "role": "user",
            "content": preguntas[0]
        }
    ]
)
end_time = time.time()
duracion = end_time - start_time

# Guardando el resultado
resultado_1 = {
    "respuesta" : response.choices[0].message.content,
    "tiempo_respuesta" : f"{duracion:.2f} s",
    "tokens_totales" : response.usage.total_tokens
}

In [ ]:
resultado_1

{'respuesta': 'En Scrum, los tres roles principales son el Product Owner, el Equipo Scrum (también llamado equipo de trabajo) y el Entidad que Aplica Scrum (EAS, en su abreviatura en inglés) o Gerente de Proceso. Tu nube de palabras indica que no distingues entre ellos, por lo que voy a describir a cada uno:\n\n1. Product Owner: Este rol se encarga de dar voz al cliente o a las personas que usarán la productividad generada por el equipo. El Product Owner es quien mantiene el Product Backlog ordenado y actualizado, garantiza que las prioritizaciones son correctas y se entiende del equipo que trabaja con el Product Backlog y se asegura de que se encuentren las necesidades del público المستهدف. También es el responsable de mezclar todas las metas de los productos y dividirlas en subtareas que pueden ser implementadas en sus pequeños incrementos o release.\n\n2. Equipo de Scrum (equipo de trabajo): Este equipo se encuentra dentro del ciclo acción-observación-comentario (A-O-C) y tiene como

**3. Repite para la segunda y tercera pregunta:** Crea `resultado_2` y `resultado_3` de la misma forma.

In [ ]:
# Consultar la segunda pregunta y guardar el resultado en resultado_2
start_time = time.time()
response = client.chat.completions.create(
    model=modelos.get("modelo_ligero"),
    messages=[
        {
            "role": "user",
            "content": preguntas[1]
        }
    ]
)
end_time = time.time()
duracion = end_time - start_time

# Guardando el resultado
resultado_2 = {
    "respuesta" : response.choices[0].message.content,
    "tiempo_respuesta" : f"{duracion:.2f} s",
    "tokens_totales" : response.usage.total_tokens
}

In [ ]:
# Consultar la tercera pregunta y guardar el resultado en resultado_3
start_time = time.time()
response = client.chat.completions.create(
    model=modelos.get("modelo_ligero"),
    messages=[
        {
            "role": "user",
            "content": preguntas[2]
        }
    ]
)
end_time = time.time()
duracion = end_time - start_time

# Guardando el resultado
resultado_3 = {
    "respuesta" : response.choices[0].message.content,
    "tiempo_respuesta" : f"{duracion:.2f} s",
    "tokens_totales" : response.usage.total_tokens
}

**4. Junta los resultados:** Agrega `resultado_1`, `resultado_2` y `resultado_3` a una lista vacía llamada `resultados`.

In [ ]:
# Definir la lista resultados y agregar los tres diccionarios
resultados = []
resultados.append(resultado_1)
resultados.append(resultado_2)
resultados.append(resultado_3)

**5. Muestra la comparación:** Imprime `resultados` y concluye si el modelo ligero resolvió las 3 preguntas satisfactoriamente.

In [ ]:
# Mostrar la tabla final de resultados
for i, resultado in enumerate(resultados, start=1):
    print(f"--- Resultado {i} ---")
    print(resultado["respuesta"])
    print()

--- Resultado 1 ---
En Scrum, los tres rol principales son el equipo de desarrollo, el Product Owner y el Scrum Master. Todos estos roles son clave para la ejecución correcta de Scrum y son responsables de roles diferentes. Aquí tienes explicación de cada uno:

1. Equipo de Desarrollo (Development Team): Este es el grupo de ingenieros, desarrolladores y profesionales asociados responsable de crear el producto. Esta es la unidad de trabajo que implementa el backlog de producto y sprint backlog. Su responsabilidad principal es crear y ofrecer un producto funcional en cada iteración que sigue Scrum.

2. Product Owner: El Product Owner es una persona designada que representa los intereses de los usuarios y los clientes. Es quien determina qué características y funciones se incluyen en el producto a partir de los requisitos y valiosos lugares de los usuarios y clientes. Cuenta con la responsabilidad de mejorar el backlog de producto colocando prioridades y proporcionando detalles suficiente

comentarios: Definitivamente el modelo ligero si resolvio las preguntas en forma satisfactoria